# Quantum Trader — DPO Dry Run (PATCH-10A)

**Før du starter: Runtime → Change runtime type → GPU (A100 anbefalt, T4 fungerer)**

---

## Gjør dette nå — i rekkefølge

| Steg | Celle | Hva skjer |
|------|-------|-----------|
| 1 | Celle 2 | Verifiser GPU og VRAM |
| 2 | Celle 3 | Installer avhengigheter |
| 3 | Celle 4 | Logg inn på Hugging Face |
| 4 | Celle 5 | Last opp datasettet og verifiser SHA256 |
| 5 | Celle 6 | Formatter DPO-datasett (80/20 split) |
| 6 | Celle 7 | **Kjør DPO-trening** (~20–35 min) |
| 7 | Celle 8 | Se loss-kurver |
| 8 | Celle 9 | Shadow eval: DPO-modell vs base |
| 9 | Celle 10 | Go / No-Go gate |
| 10 | Celle 11 | Last ned adapter (bare hvis GO) |

---

**Dataset:** 251 par · SHA256 `5fb175f398e9b87846559d0867351644c25e63ae5f46cdcd118ed311ea5ced80`  
**Base model:** `meta-llama/Meta-Llama-3.1-8B-Instruct` (8B QLoRA proxy)  
**Output:** `dpo_run_output/` — LoRA adapter, eval_results.json  
**Ingen live-bruk** — adapter testes i shadow på testnet etter GO-signal.

In [ ]:
# Celle 2 — Verifiser GPU og VRAM
import subprocess, sys, torch

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('Ingen GPU funnet!\nGå til:  Runtime → Change runtime type → GPU')

gpu_info = result.stdout.strip()
print('GPU:', gpu_info)

# Parse VRAM (MiB)
try:
    parts   = gpu_info.split(',')
    vram_mb = int(parts[1].strip().replace(' MiB', ''))
    print(f'VRAM: {vram_mb / 1024:.0f} GB')
    if vram_mb < 16_000:
        print('ADVARSEL: <16 GB VRAM — reduser BATCH_SIZE til 1 og GRAD_ACCUM til 16 i Celle 7')
    elif vram_mb < 24_000:
        print('OK: T4/16 GB — kjørbar med 4-bit QLoRA')
    else:
        print('OK: ≥24 GB — full 4-bit QLoRA med batch_size=4')
except Exception:
    print('Kunne ikke parse VRAM — fortsett likevel')

print(f'\ntorch: {torch.__version__}')
print(f'CUDA:  {torch.version.cuda}')
print(f'bf16:  {torch.cuda.is_bf16_supported()}')

In [ ]:
# Cell 2 — Install requirements
!pip install -q \
    'torch>=2.2.0' \
    'transformers>=4.44.0' \
    'trl>=0.9.4' \
    'peft>=0.11.0' \
    'datasets>=2.20.0' \
    'accelerate>=0.31.0' \
    'bitsandbytes>=0.43.0' \
    'sentencepiece' \
    'protobuf'
print('Requirements installed.')

In [ ]:
# Celle 4 — Hugging Face login
# ─────────────────────────────────────────────────────────────────────────────
# Tips: legg tokenet inn i Colab Secrets i stedet for å lime det inn her.
#   Venstre panel → 🔑-ikon → "Add new secret" → navn: HF_TOKEN
# ─────────────────────────────────────────────────────────────────────────────
# Du trenger lese-tilgang til:  meta-llama/Meta-Llama-3.1-8B-Instruct
# Godkjenn Llama 3.1-lisensen på:
#   https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct
from huggingface_hub import login
import os

# Prøv Colab Secrets først
hf_token = ''
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN') or ''
except Exception:
    pass

if not hf_token:
    hf_token = os.environ.get('HF_TOKEN', '')

if not hf_token:
    hf_token = input('Lim inn Hugging Face-token (lagres ikke): ').strip()

login(token=hf_token)
print('Innlogget på Hugging Face.')

In [ ]:
# Celle 5 — Last inn og verifiser datasettet
# ─────────────────────────────────────────────────────────────────────────────
# Velg én metode — kjøres automatisk i prioritert rekkefølge:
#   A) Google Drive (anbefalt hvis filen allerede ligger der)
#   B) Direkte filopplasting via Colab
# ─────────────────────────────────────────────────────────────────────────────
import hashlib, pathlib, shutil

EXPECTED_SHA = '5fb175f398e9b87846559d0867351644c25e63ae5f46cdcd118ed311ea5ced80'
FNAME        = 'dpo_patch10a_v2_FROZEN.jsonl'

# ── Metode A: Google Drive ────────────────────────────────────────────────────
# Legg filen i Google Drive og oppdater stien under.
GDRIVE_PATH = '/content/drive/MyDrive/quantum_trader/dpo_patch10a_v2_FROZEN.jsonl'

_loaded = False

if not _loaded and not pathlib.Path(FNAME).exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        if pathlib.Path(GDRIVE_PATH).exists():
            shutil.copy(GDRIVE_PATH, FNAME)
            print(f'Lastet fra Google Drive: {GDRIVE_PATH}')
            _loaded = True
        else:
            print(f'Filen finnes ikke i Drive på {GDRIVE_PATH}')
            print('Prøver filopplasting i stedet ...')
    except Exception as e:
        print(f'Google Drive ikke tilgjengelig ({e}) — prøver filopplasting ...')

# ── Metode B: Direkte filopplasting ──────────────────────────────────────────
if not _loaded and not pathlib.Path(FNAME).exists():
    from google.colab import files
    print(f'Last opp {FNAME} når dialogboksen åpner seg ...')
    uploaded = files.upload()
    src = list(uploaded.keys())[0]
    if src != FNAME:
        shutil.copy(src, FNAME)
    _loaded = True

# ── SHA256-verifisering ───────────────────────────────────────────────────────
actual_sha = hashlib.sha256(pathlib.Path(FNAME).read_bytes()).hexdigest()
if actual_sha != EXPECTED_SHA:
    raise ValueError(
        f'SHA256-feil!\n  Forventet: {EXPECTED_SHA}\n  Faktisk:   {actual_sha}\n'
        'Kontroller at du bruker den frosne filen fra VPS.')

lines = pathlib.Path(FNAME).read_text().count('\n')
print(f'\n✓ Dataset OK: {lines} par, SHA256 verifisert.')

In [ ]:
# Cell 5 — Inline format_dpo_dataset.py (so the notebook is self-contained)
import json, pathlib, random

SYSTEM_PROMPT = (
    'You are a constrained trading risk advisor operating on Binance testnet futures.\n'
    'You receive a JSON object describing one open position and the formula engine\'s\n'
    'exit recommendation. Your only job is to select an exit action.\n\n'
    'Allowed actions (you MUST use exactly one of these strings):\n'
    '  HOLD\n'
    '  PARTIAL_CLOSE_25\n'
    '  FULL_CLOSE\n'
    '  TIME_STOP_EXIT\n\n'
    'Rules:\n'
    '- Emergency stops are already handled upstream. You will never see urgency=EMERGENCY.\n'
    '- TIGHTEN_TRAIL and MOVE_TO_BREAKEVEN are not available to you.\n'
    '- If uncertain, prefer HOLD (defer to the formula recommendation).\n'
    '- formula_suggestion is advisory. You may agree or override it.\n\n'
    'You MUST respond with ONLY a JSON object — no markdown, no explanation, no extra text:\n'
    '{"action": "<one of the 4 actions>", "confidence": <0.0-1.0>, "reason": "<max 120 chars>"}'
)

_CHOSEN_REASONS = {
    ('premature_close',   'HOLD'):             'Position closed too early — insufficient hold time, wait for stronger signal',
    ('late_hold',         'FULL_CLOSE'):        'Exit score strong and hold overdue — close fully now to capture remaining profit',
    ('late_hold',         'PARTIAL_CLOSE_25'):  'Moderate exit signal with long hold — partial close to reduce exposure',
    ('divergence_regret', 'FULL_CLOSE'):        'Formula recommended full close; override reverses the divergence loss',
    ('divergence_regret', 'PARTIAL_CLOSE_25'):  'Formula recommended partial close; reversing Qwen3 override to follow formula',
    ('none',              'PARTIAL_CLOSE_25'):  'Reward negative and exit signal moderate — partial exit de-risks position',
    ('none',              'FULL_CLOSE'):        'Reward strongly negative — exit fully to protect capital',
    ('none',              'HOLD'):              'No strong signal; hold and continue monitoring',
}
_REJECTED_REASONS = {
    'HOLD':             'Formula says hold; deferring to formula recommendation',
    'PARTIAL_CLOSE_25': 'Moderate exit score; partial close to lock some profit',
    'FULL_CLOSE':       'High exit score; closing full position',
    'TIME_STOP_EXIT':   'Max hold time reached; exiting by time stop',
}
_ACTION_CONF = {'HOLD': 0.65, 'PARTIAL_CLOSE_25': 0.72, 'FULL_CLOSE': 0.80, 'TIME_STOP_EXIT': 0.60}

def _build_user_payload(pair):
    es = float(pair.get('exit_score') or 0.5)
    fa = pair.get('formula_action') or 'HOLD'
    return json.dumps({'symbol': pair['symbol'], 'side': pair.get('side', 'LONG'),
                       'exit_score': round(es, 4),
                       'formula_suggestion': {'action': fa, 'confidence': round(min(1.0, es+0.1), 4),
                                              'reason': 'formula engine output'}},
                      separators=(',', ':'))

def build_example(pair):
    preferred = pair['preferred_action']
    live      = pair['live_action']
    regret    = pair.get('regret_label', 'none')
    prompt = (f'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n'
              f'{SYSTEM_PROMPT}\n'
              f'<|eot_id|><|start_header_id|>user<|end_header_id|>\n'
              f'{_build_user_payload(pair)}\n'
              f'<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n')
    chosen  = json.dumps({'action': preferred, 'confidence': _ACTION_CONF.get(preferred, 0.7),
                          'reason': _CHOSEN_REASONS.get((regret, preferred), f'{regret}: {preferred} is better')})
    rejected = json.dumps({'action': live, 'confidence': _ACTION_CONF.get(live, 0.65),
                           'reason': _REJECTED_REASONS.get(live, f'agent selected {live}')})
    return {'prompt': prompt, 'chosen': chosen, 'rejected': rejected}

pairs = [json.loads(l) for l in pathlib.Path('dpo_patch10a_v2_FROZEN.jsonl').open() if l.strip()]
random.seed(42)
random.shuffle(pairs)
n_val = max(40, int(len(pairs) * 0.20))
n_train = len(pairs) - n_val
train_p, val_p = pairs[:n_train], pairs[n_train:]

pathlib.Path('dpo_formatted').mkdir(exist_ok=True)
with open('dpo_formatted/dpo_train.jsonl', 'w') as f:
    for p in train_p: f.write(json.dumps(build_example(p)) + '\n')
with open('dpo_formatted/dpo_val.jsonl', 'w') as f:
    for p in val_p:   f.write(json.dumps(build_example(p)) + '\n')

print(f'Train: {n_train} examples  Val: {n_val} examples')
ex = build_example(pairs[0])
print('\nSample prompt (last 200 chars):', ex['prompt'][-200:])
print('Sample chosen:', ex['chosen'][:120])

In [ ]:
# Celle 7 — DPO-trening
# ─────────────────────────────────────────────────────────────────────────────
# Juster bare disse to linjene ved behov:
#   BATCH_SIZE  →  sett til 1 hvis T4 (16 GB VRAM) gir OOM
#   GRAD_ACCUM  →  øk tilsvarende slik at effektiv batch = 16
# ─────────────────────────────────────────────────────────────────────────────
import json, os, pathlib, torch
from datasets import Dataset
from peft import LoraConfig, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import DPOConfig, DPOTrainer

BASE_MODEL     = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
LORA_RANK      = 8
LORA_ALPHA     = 16
DPO_BETA       = 0.5
LEARNING_RATE  = 5e-5
BATCH_SIZE     = 4    # ← sett til 1 ved OOM på T4
GRAD_ACCUM     = 4    # ← sett til 16 ved BATCH_SIZE=1
NUM_EPOCHS     = 2
MAX_SEQ_LEN    = 1024
MAX_PROMPT_LEN = 700  # system prompt ≈ 310 tokens — must be > this
EVAL_STEPS     = 20

def load_jsonl(path):
    return Dataset.from_list([json.loads(l) for l in open(path) if l.strip()])

# bf16 on A100; fp16 on T4/V100
_use_bf16 = torch.cuda.is_bf16_supported()
print(f'Precision: {"bf16" if _use_bf16 else "fp16"}')

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if _use_bf16 else torch.float16,
    bnb_4bit_use_double_quant=True)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

_attn_kwargs = {}
try:
    import importlib; importlib.import_module('flash_attn')
    _attn_kwargs['attn_implementation'] = 'flash_attention_2'
    print('flash_attn funnet — bruker flash_attention_2')
except ImportError:
    print('flash_attn ikke installert — bruker eager attention (OK)')

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_cfg, device_map='auto',
    torch_dtype=torch.bfloat16 if _use_bf16 else torch.float16,
    **_attn_kwargs)
model.config.use_cache = False
model.enable_input_require_grads()

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=LORA_RANK, lora_alpha=LORA_ALPHA,
    lora_dropout=0.05, bias='none', inference_mode=False,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])

train_ds = load_jsonl('dpo_formatted/dpo_train.jsonl')
val_ds   = load_jsonl('dpo_formatted/dpo_val.jsonl')
print(f'Datasett: train={len(train_ds)}  val={len(val_ds)}')

pathlib.Path('dpo_run_output/logs').mkdir(parents=True, exist_ok=True)
dpo_args = DPOConfig(
    output_dir='dpo_run_output', num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine', warmup_ratio=0.1, weight_decay=0.01,
    beta=DPO_BETA, max_length=MAX_SEQ_LEN, max_prompt_length=MAX_PROMPT_LEN,
    eval_strategy='steps', eval_steps=EVAL_STEPS,
    save_strategy='steps', save_steps=40, save_total_limit=2,
    logging_dir='dpo_run_output/logs', logging_steps=5,
    fp16=not _use_bf16, bf16=_use_bf16, report_to='none',
    load_best_model_at_end=True, metric_for_best_model='eval_loss',
    greater_is_better=False, remove_unused_columns=False,
)

trainer = DPOTrainer(
    model=model, ref_model=None, args=dpo_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    tokenizer=tokenizer, peft_config=lora_cfg,
)
trainer.train()
trainer.save_model('dpo_run_output')
tokenizer.save_pretrained('dpo_run_output')
metrics = trainer.evaluate()
with open('dpo_run_output/eval_results.json', 'w') as f: json.dump(metrics, f, indent=2)
print('\nTrening ferdig.')
print('Eval metrics:', metrics)

In [ ]:
# Celle 8 — Loss-kurver
import json, pathlib

ts   = json.loads(pathlib.Path('dpo_run_output/trainer_state.json').read_text())
logs = ts.get('log_history', [])

train_steps = [(e['step'], e['loss'])      for e in logs if 'loss'      in e and 'eval_loss' not in e]
eval_steps  = [(e['step'], e['eval_loss']) for e in logs if 'eval_loss' in e]

if train_steps:
    t_x, t_y = zip(*train_steps)
    print(f'Train loss:  {t_y[0]:.4f}  →  {t_y[-1]:.4f}  (delta={t_y[-1]-t_y[0]:+.4f})')
if eval_steps:
    e_x, e_y = zip(*eval_steps)
    best_val  = min(e_y)
    final_val = e_y[-1]
    ratio     = final_val / best_val if best_val > 0 else 999
    print(f'Val loss:    {e_y[0]:.4f}  →  {final_val:.4f}  (best={best_val:.4f}, ratio={ratio:.2f})')
    if ratio > 1.25:
        print('  ⚠️  Val loss steg til >1.25× best — overfitting')
    else:
        print('  ✓ Val loss stabil')

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(9, 4))
    if train_steps: ax.plot(t_x, t_y, label='train loss', linewidth=1.5)
    if eval_steps:
        ax.plot(e_x, e_y, label='val loss', marker='o', linewidth=1.5)
        ax.axhline(best_val, color='gray', linestyle='--', linewidth=0.8, label=f'best val={best_val:.4f}')
    ax.set_xlabel('step'); ax.set_ylabel('loss')
    ax.set_title('DPO Training — Loss Curves (PATCH-10A)')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
except Exception as exc:
    print(f'Plot ikke tilgjengelig: {exc}')

In [ ]:
# Celle 9 — Shadow eval: DPO-tuned vs base
# Modellene lastes én om gangen for å unngå OOM.
import gc, json, pathlib, torch
from collections import Counter
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

val_pairs = [json.loads(l) for l in open('dpo_formatted/dpo_val.jsonl') if l.strip()]
print(f'Evaluerer {len(val_pairs)} valideringpar ...')

BASE = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
_use_bf16 = torch.cuda.is_bf16_supported()

def _run_inference(adapter_path):
    """Load one model, run all val pairs, unload, return list of (action_str,)."""
    tok = AutoTokenizer.from_pretrained(BASE, use_fast=True)
    tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(
        BASE, torch_dtype=torch.bfloat16 if _use_bf16 else torch.float16,
        device_map='auto')
    if adapter_path:
        m = PeftModel.from_pretrained(m, adapter_path)
    gen = pipeline('text-generation', model=m, tokenizer=tok,
                   max_new_tokens=80, do_sample=False, temperature=1.0)
    actions = []
    for row in val_pairs:
        raw = (gen(row['prompt'], return_full_text=False)[0]['generated_text'] or '').strip()
        try:    actions.append(json.loads(raw).get('action', 'PARSE_ERROR'))
        except: actions.append('PARSE_ERROR')
    # Free GPU memory
    del gen, m
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return actions

print('Kjører base-modell (ingen adapter) ...')
base_actions = _run_inference(None)

print('Kjører DPO-modell (med adapter) ...')
tuned_actions = _run_inference('dpo_run_output')

# Compute metrics
preferred_actions = [json.loads(r['chosen']).get('action', '') for r in val_pairs]
n = len(val_pairs)
base_correct  = sum(b == p for b, p in zip(base_actions,  preferred_actions))
tuned_correct = sum(t == p for t, p in zip(tuned_actions, preferred_actions))
ba   = base_correct  / n
ta   = tuned_correct / n
lift = ta - ba
base_dist  = Counter(base_actions)
tuned_dist = Counter(tuned_actions)
pe_tuned   = tuned_dist.get('PARSE_ERROR', 0) / n

shadow = {
    'n': n, 'base_accuracy': ba, 'tuned_accuracy': ta, 'lift': lift,
    'base_dist': dict(base_dist), 'tuned_dist': dict(tuned_dist),
    'preferred_dist': dict(Counter(preferred_actions)),
    'parse_error_base': base_dist.get('PARSE_ERROR', 0) / n,
    'parse_error_tuned': pe_tuned,
    'per_pair': [{'preferred': p, 'base': b, 'tuned': t,
                  'base_correct': b == p, 'tuned_correct': t == p}
                 for p, b, t in zip(preferred_actions, base_actions, tuned_actions)],
}
with open('shadow_eval_results.json', 'w') as f: json.dump(shadow, f, indent=2)

print(f'\nBase accuracy:  {ba:.3f}  ({base_correct}/{n})')
print(f'Tuned accuracy: {ta:.3f}  ({tuned_correct}/{n})')
print(f'Accuracy lift:  {lift:+.3f}')
print(f'Parse errors:   {pe_tuned:.3f}')
print('Tuned dist:', dict(tuned_dist))
print('\nResultater → shadow_eval_results.json')

In [ ]:
# Cell 8 — Go / No-Go gate (inline version of go_no_go.py)
import json, pathlib, sys

THRESHOLDS = {'min_loss_reduction': 0.05, 'max_val_loss_ratio': 1.25,
              'min_tuned_accuracy': 0.40, 'min_accuracy_lift': 0.05,
              'max_parse_error_rate': 0.05, 'max_single_action_share': 0.80,
              'forbidden_time_stop_share': 0.20}

failures = []

# --- Training convergence ---
ts = json.loads(pathlib.Path('dpo_run_output/trainer_state.json').read_text())
log = ts.get('log_history', [])
train_steps = [e for e in log if 'loss' in e and 'eval_loss' not in e]
eval_steps  = [e for e in log if 'eval_loss' in e]
if train_steps:
    reduction = train_steps[0]['loss'] - train_steps[-1]['loss']
    if reduction < THRESHOLDS['min_loss_reduction']:
        failures.append(f'train_loss_reduction={reduction:.4f} < {THRESHOLDS["min_loss_reduction"]}')
    else:
        print(f'[PASS] train_loss_reduction={reduction:.4f}')
if eval_steps:
    best_val  = min(e['eval_loss'] for e in eval_steps)
    final_val = eval_steps[-1]['eval_loss']
    ratio     = final_val / best_val if best_val > 0 else 999
    if ratio > THRESHOLDS['max_val_loss_ratio']:
        failures.append(f'val_loss_ratio={ratio:.2f} > {THRESHOLDS["max_val_loss_ratio"]}')
    else:
        print(f'[PASS] val_loss_ratio={ratio:.2f}  best={best_val:.4f} final={final_val:.4f}')

# --- Shadow eval ---
se = json.loads(pathlib.Path('shadow_eval_results.json').read_text())
ta, ba, lift = se['tuned_accuracy'], se['base_accuracy'], se['lift']
pe = se['parse_error_tuned']
dist = se['tuned_dist']
total = sum(dist.values())

checks = [
    ('min_tuned_accuracy',      ta >= THRESHOLDS['min_tuned_accuracy'],      f'tuned_accuracy={ta:.3f}'),
    ('min_accuracy_lift',       lift >= THRESHOLDS['min_accuracy_lift'],      f'lift={lift:+.3f}'),
    ('max_parse_error_rate',    pe <= THRESHOLDS['max_parse_error_rate'],     f'parse_error={pe:.3f}'),
    ('action_diversity',        max(dist.values())/total <= THRESHOLDS['max_single_action_share'] if total else True,
                                f'dominant={max(dist,key=dist.get)} share={max(dist.values())/total:.2f}' if total else 'empty'),
]
for name, passed, detail in checks:
    sym = 'PASS' if passed else 'FAIL'
    print(f'[{sym}] {name}: {detail}')
    if not passed:
        failures.append(f'{name}: {detail}')

print()
print('─' * 50)
if not failures:
    print('✅  VERDICT: GO')
    print('Adapter is ready for shadow validation on Binance testnet.')
    print('Next: download dpo_run_output/ and copy to VPS.')
else:
    print('❌  VERDICT: NO-GO')
    for f in failures:
        print(f'  FAIL: {f}')
    print()
    print('Recommended fixes:')
    if any('loss_reduction' in f or 'val_loss_ratio' in f for f in failures):
        print('  → Training diverged or overfit. Try DPO_BETA=0.1 or NUM_EPOCHS=1.')
    if any('accuracy' in f or 'lift' in f for f in failures):
        print('  → Model did not learn. Check system prompt match in Cell 5.')
    if any('parse' in f for f in failures):
        print('  → JSON format decayed. Try lower learning rate (1e-5) or fewer epochs.')

In [ ]:
# Cell 9 — Download adapter (only run if verdict was GO)
import subprocess, pathlib
from google.colab import files

subprocess.run(['tar', '-czf', 'dpo_run_output.tar.gz', 'dpo_run_output/'], check=True)
size_mb = pathlib.Path('dpo_run_output.tar.gz').stat().st_size / 1_048_576
print(f'Packed adapter: dpo_run_output.tar.gz  ({size_mb:.1f} MB)')
files.download('dpo_run_output.tar.gz')
print('Download started.')
print()
print('After download, copy to VPS:')
print('  scp -i ~/.ssh/hetzner_fresh dpo_run_output.tar.gz root@46.224.116.254:/home/qt/quantum_trader/ops/offline/')
print('  ssh ... "cd /home/qt/quantum_trader/ops/offline && tar -xzf dpo_run_output.tar.gz"')
print()
print('Then enable shadow mode on testnet:')
print('  systemctl edit qt-exit-agent --force')
print('  [Service]')
print('  Environment="EXIT_AGENT_QWEN3_SHADOW=true"')
print('  Environment="EXIT_AGENT_DPO_ADAPTER=/home/qt/quantum_trader/ops/offline/dpo_run_output"')
print('  Environment="BINANCE_TESTNET=true"')